# Where Models Run: Cloud, Device, and Routing Between Models

The runnable code of [Where Models Run: Cloud, Device, and Routing Between Models](https://residual-stream.lvtan71.workers.dev/posts/where-models-run-and-routing/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/where-models-run-and-routing.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 2 min end to end.


The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "where-models-run-and-routing"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/where-models-run-and-routing/route.py']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## A cascade, measured

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/where-models-run-and-routing/#a-cascade-measured)

In [ ]:
import json
import random
import re
import time
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import snapshot_download
GPU = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"  # notebook: the post ran on "mps"
gpu_sync = {"cuda": torch.cuda.synchronize, "mps": torch.mps.synchronize}.get(GPU, lambda: None)
gpu_empty_cache = {"cuda": torch.cuda.empty_cache, "mps": torch.mps.empty_cache}.get(GPU, lambda: None)

# Routing between a small and a large model. Small: Qwen2.5-1.5B-Instruct;
# large: Qwen2.5-3B-Instruct (stand-ins for an on-device model and a bigger
# served one). 150 questions with checkable answers, of three kinds: capital
# cities, two-digit multiplications, and day-of-week arithmetic. Each model
# answers every question greedily; the small model's confidence is the
# probability it assigned to its own answer's tokens. A cascade sends a
# question to the large model when the small one's confidence is below a
# threshold. Cost: each model's measured time per question on this machine.
PATHS = {"small": snapshot_download("Qwen/Qwen2.5-1.5B-Instruct",
                                    revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306"),
         "large": snapshot_download("Qwen/Qwen2.5-3B-Instruct",
                                    revision="aa8e72537993ba99e69dfaafa59ed015b17504d1")}
rng = random.Random(0)
caps = [("France", "Paris"), ("Japan", "Tokyo"), ("Canada", "Ottawa"), ("Australia", "Canberra"), ("Brazil", "Brasilia"),
        ("Turkey", "Ankara"), ("Switzerland", "Bern"), ("Nigeria", "Abuja"), ("Vietnam", "Hanoi"), ("Kenya", "Nairobi"),
        ("Morocco", "Rabat"), ("Pakistan", "Islamabad"), ("Myanmar", "Naypyidaw"), ("Kazakhstan", "Astana"), ("Peru", "Lima"),
        ("Egypt", "Cairo"), ("Norway", "Oslo"), ("Chile", "Santiago"), ("Tanzania", "Dodoma"), ("Bolivia", "Sucre"),
        ("Sri Lanka", "Colombo"), ("Ivory Coast", "Yamoussoukro"), ("Germany", "Berlin"), ("Italy", "Rome"), ("Spain", "Madrid"),
        ("Poland", "Warsaw"), ("Ethiopia", "Addis Ababa"), ("Ghana", "Accra"), ("Mongolia", "Ulaanbaatar"), ("Laos", "Vientiane"),
        ("Benin", "Porto-Novo"), ("Belize", "Belmopan"), ("Bhutan", "Thimphu"), ("Malawi", "Lilongwe"), ("Burundi", "Gitega"),
        ("Palau", "Ngerulmud"), ("Eswatini", "Mbabane"), ("Iceland", "Reykjavik"), ("Cuba", "Havana"), ("Nepal", "Kathmandu"),
        ("Uganda", "Kampala"), ("Zambia", "Lusaka"), ("Oman", "Muscat"), ("Qatar", "Doha"), ("Jordan", "Amman"),
        ("Latvia", "Riga"), ("Estonia", "Tallinn"), ("Slovenia", "Ljubljana"), ("Malta", "Valletta"), ("Fiji", "Suva")]
DAYS = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
items = [(f"What is the capital of {c}? Answer with just the city name.", a, "capital") for c, a in caps]
for _ in range(50):
    a, b = rng.randint(11, 99), rng.randint(11, 99)
    items.append((f"What is {a} times {b}? Answer with just the number.", str(a * b), "multiplication"))
for _ in range(50):
    d, n = rng.randrange(7), rng.randint(2, 60)
    items.append((f"Today is {DAYS[d]}. What day of the week will it be in {n} days? Answer with just the day.",
                  DAYS[(d + n) % 7], "weekday"))

@torch.no_grad()
def answer_all(path):
    tok = AutoTokenizer.from_pretrained(path)
    model = AutoModelForCausalLM.from_pretrained(path, dtype=torch.float16).to(GPU).eval()
    out = []
    for q, truth, kind in items:
        ids = tok.apply_chat_template([{"role": "user", "content": q}], add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True)["input_ids"].to(GPU)
        gpu_sync(); t0 = time.perf_counter()
        g = model.generate(ids, max_new_tokens=12, do_sample=False, temperature=None, top_p=None, top_k=None,
                           output_scores=True, return_dict_in_generate=True)
        gpu_sync(); dt = time.perf_counter() - t0
        new = g.sequences[0, ids.shape[1]:]
        logp = [torch.log_softmax(s[0].float(), -1)[t].item() for s, t in zip(g.scores, new)]
        text = tok.decode(new, skip_special_tokens=True).strip()
        right = re.sub(r"[^a-z0-9]", "", truth.lower()) in re.sub(r"[^a-z0-9]", "", text.lower())
        out.append({"right": right, "conf": float(np.exp(np.mean(logp[:max(1, len(logp) - 1)]))), "time": dt, "text": text})
    del model; gpu_empty_cache()
    return out

res = {name: answer_all(p) for name, p in PATHS.items()}
json.dump(res, open("answers.json", "w"))
S, L = res["small"], res["large"]
ts, tl = np.mean([r["time"] for r in S]), np.mean([r["time"] for r in L])
print(f"{len(items)} questions; seconds per question: small {ts:.2f}, large {tl:.2f}\n")
print(f"{'':10s} {'small':>7s} {'large':>7s}   (accuracy by kind)")
for kind in ("capital", "multiplication", "weekday"):
    idx = [i for i, it in enumerate(items) if it[2] == kind]
    print(f"{kind:14s} {np.mean([S[i]['right'] for i in idx]):5.0%} {np.mean([L[i]['right'] for i in idx]):7.0%}")
print(f"{'all':14s} {np.mean([r['right'] for r in S]):5.0%} {np.mean([r['right'] for r in L]):7.0%}\n")

# The cascade: every question goes to the small model first; below the
# confidence threshold it is also sent to the large model, whose answer is used.
print(f"{'policy':34s} {'accuracy':>9s} {'sent to large':>14s} {'seconds/question':>17s}")
def report(name, use_large):
    acc = np.mean([L[i]["right"] if u else S[i]["right"] for i, u in enumerate(use_large)])
    cost = ts + np.mean(use_large) * tl if "cascade" in name or "oracle" in name else (tl if all(use_large) else ts)
    print(f"{name:34s} {acc:9.0%} {np.mean(use_large):14.0%} {cost:17.2f}")
report("always small", [False] * len(items))
report("always large", [True] * len(items))
for th in (0.5, 0.7, 0.8, 0.9, 0.95):
    report(f"cascade, escalate below {th:.2f}", [r["conf"] < th for r in S])
report("oracle cascade (escalate if wrong)", [not r["right"] for r in S])
right = np.array([r["right"] for r in S]); conf = np.array([r["conf"] for r in S])
print(f"\nsmall model: confidence of right answers median {np.median(conf[right]):.2f}, "
      f"of wrong answers {np.median(conf[~right]):.2f}")

The post printed:

```text
150 questions; seconds per question: small 0.22, large 0.40

             small   large   (accuracy by kind)
capital          92%     88%
multiplication   54%     84%
weekday          18%     26%
all              55%     66%

policy                              accuracy  sent to large  seconds/question
always small                             55%             0%              0.22
always large                             66%           100%              0.40
cascade, escalate below 0.50             57%             7%              0.25
cascade, escalate below 0.70             60%            19%              0.29
cascade, escalate below 0.80             59%            30%              0.34
cascade, escalate below 0.90             63%            43%              0.39
cascade, escalate below 0.95             65%            55%              0.44
oracle cascade (escalate if wrong)       73%            45%              0.40

small model: confidence of right answers median 0.99, of wrong answers 0.78
```

In [ ]:
import json
import re
import unicodedata
import numpy as np

# Rescoring route.py's answers (answers.json) and looking inside the cascade.
# route.py's own table used a matcher that dropped accented letters
# ("Brasília" failed) and accepted one capital per country; here accents are
# folded and the documented alternatives (La Paz for Bolivia, Nur-Sultan for
# Kazakhstan's capital 2019-2022, Bujumbura for Burundi's former capital) count.
exec(open("route.py").read().split("@torch.no_grad()")[0].split("PATHS = ")[0])   # imports only
src = open("route.py").read()
exec(src[src.index("rng = random.Random(0)"):src.index("@torch.no_grad()")])        # the same 150 items
res = json.load(open("answers.json"))
ALSO = {"Sucre": ["la paz"], "Astana": ["nur-sultan", "nursultan"], "Gitega": ["bujumbura"]}
fold = lambda t: re.sub(r"[^a-z0-9]", "", unicodedata.normalize("NFKD", t.lower()).encode("ascii", "ignore").decode())
def right(text, truth):
    return any(fold(a) in fold(text) for a in [truth] + ALSO.get(truth, []))
S = np.array([right(r["text"], it[1]) for r, it in zip(res["small"], items)])
L = np.array([right(r["text"], it[1]) for r, it in zip(res["large"], items)])
conf = np.array([r["conf"] for r in res["small"]])
kind = np.array([it[2] for it in items])
print(f"{'':16s} {'small':>6s} {'large':>6s}")
for k in ("capital", "multiplication", "weekday"):
    m = kind == k
    print(f"{k:16s} {S[m].mean():6.0%} {L[m].mean():6.0%}   small's confidence when wrong: median {np.median(conf[m & ~S]):.2f}")
print(f"{'all':16s} {S.mean():6.0%} {L.mean():6.0%}\n")

rng = np.random.default_rng(0)
def report(name, esc):
    acc = np.where(esc, L, S)
    boot = [np.where(esc, L, S)[i].mean() - S[i].mean() for i in (rng.integers(0, len(S), len(S)) for _ in range(2000))]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    by = ", ".join(f"{k} {int((esc & (kind == k)).sum())}" for k in ("capital", "multiplication", "weekday"))
    print(f"{name:40s} {acc.mean():5.0%}  gain over small {acc.mean() - S.mean():+5.0%} [{lo:+.0%}, {hi:+.0%}]  escalated: {by}")
for th in (0.7, 0.9):
    report(f"cascade, escalate below {th}", conf < th)
report("route by type: only capitals to small", kind != "capital")
report("route by type: only multiplication to large", kind == "multiplication")
report("always large", np.ones(len(S), bool))
report("oracle (escalate exactly when small is wrong)", ~S)

The post printed:

```text
                  small  large
capital             96%    94%   small's confidence when wrong: median 0.78
multiplication      54%    84%   small's confidence when wrong: median 0.86
weekday             18%    26%   small's confidence when wrong: median 0.67
all                 56%    68%

cascade, escalate below 0.7                61%  gain over small   +5% [+1%, +9%]  escalated: capital 1, multiplication 1, weekday 26
cascade, escalate below 0.9                65%  gain over small   +9% [+3%, +15%]  escalated: capital 6, multiplication 17, weekday 41
route by type: only capitals to small      69%  gain over small  +13% [+6%, +19%]  escalated: capital 0, multiplication 50, weekday 50
route by type: only multiplication to large   66%  gain over small  +10% [+5%, +16%]  escalated: capital 0, multiplication 50, weekday 0
always large                               68%  gain over small  +12% [+5%, +20%]  escalated: capital 50, multiplication 50, weekday 50
oracle (escalate exactly when small is wrong)   73%  gain over small  +17% [+11%, +23%]  escalated: capital 2, multiplication 23, weekday 41
```